# Exercise 2. Build a Chatbot
In this exercise, we'll spend some time building a simple chatbot. For this, we'll be using a new, instruction-tuned "small" LLM called `SmolLM2-360M-Instruct` by Hugging Face (see their paper: {cite:t}`allal_smollm2_2025`)

## 2.1 Setup
Let's start by importing what we need (should have been installed during previous exercise!)

In [41]:
from transformers import AutoTokenizer, pipeline

## 2.2 Define a Chatbot Class
When should you use a class in Python? This can be tricky to decide. As a rule of thumb, if you do not need to *store* and *carry* information between operations, functions are usually enough.

**A chatbot is a good example of when a class is useful**, as it needs to hold information across multiple interactions. For example, a `chat_history` containing all user and LLM messages gives the chatbot *a form of memory*. See the overview below:

```{figure} ../figures/class6/chatbot-class-components.png
---
name: zero-shot-few-shot-fine-tuning overview
width: 100%
---
AI-generated, modified by me. Note that an *api_key* is only relevant for models hosted on other inference servers (e.g,. from third-party GPU cloud platforms or using OpenAI's models).
```

Let's define our *chatbot* class. Some of this may look a little new to some of you, but we'll try to break things down and slowly increase in complexity:

In [115]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------

    """
    def __init__(self, model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct"):
        self.model_id = model_id
        self.chat_history = []

    def load(self):
        pass

Let us look at `__init__()`. This special method is called when we create an instance of the class (for example, `chatbot = Chatbot(...)`) and initializes all attributes. Some attributes are provided by the user, while *internal attributes* are not!

```{figure} ../figures/class6/class_init.jpg
---
name: class_init
width: 100%
---
By me (using [carbon.sh](https://carbon.now.sh/?bg=rgba(255,255,255,0)&t=one-light&wt=none&l=python&width=824&ds=false&dsyoff=20px&dsblur=68px&wc=true&wa=false&pv=56px&ph=56px&ln=false&fl=1&fm=Hack&fs=14px&lh=133%25&es=2x&wm=false&code=class%2520Chatbot%253A%250A%2520%2520%2520%2520def%2520__init__(self%252C%2520model_id)%253A%250A%2520%2520%2520%2520%2520%2520%2520%2520self.model_id%2520%253D%2520model_id%250A%2520%2520%2520%2520%2520%2520%2520%2520self.chat_history%2520%253D%2520%255B%255D))
```

Let's add `self.model` and `self.tokenizer` and set them to `None` for now (we'll load them later!)

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------

    """
    def __init__(
        self,
        model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct",
    ):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
    
    def load(self):
        pass

### Your Turn: Add Temperature in `__init__()`
:::{admonition} HANDS-ON
:class: red
Let's do a simple addition! 
1. Add a `temperature` parameter to `__init__()` that the user can pass to `Chatbot()` (to later be used in `pipeline`). 
2. Make the parameter default to `None`, so that a user *can* but is not *forced* to specify it.
3. Remember to also specify it as an attribute

Optionally, you can specify [type hint](https://docs.python.org/3/library/typing.html) that corresponds to type of value that `temperature` can take (see [docs](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationConfig.temperature))
:::

#### Solution

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------

    """
    def __init__(
        self,
        model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct",
        temperature: float = None,
    ):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self):
        pass

## 2.3 Loading an LLM with our Chatbot
As you may have noticed, our Chatbot class has an unspecified `load` method. Let's define what this function should do (and how it should use our attributes). 

### Your Turn: Define a Load Method
:::{admonition} HANDS-ON
:class: red
Remove `pass` from `def load(self)` and start filling out the function!
1.  The load function should load the tokenizer and `pipeline` as we did in the previous exercise!
2. Add `task` as a parameter in `load` with the default `task = text_generation`
3. Remember to add defined class attributes to `pipeline` when relevant!
:::

If you are unfamiliar with class attributes, I suggest you read the little hint box before proceeding!

:::{admonition} Using attributes in methods?
:class: tip, dropdown
Imagine a Person class like this:
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color
```

When we intialise a Person class, we can access its attributes:
```python
mina = Person(name = "Mina", favorite_color = "green")
print(mina.name) # prints Mina
```

This is great! But where attributes really shine is how they are used in subsequent methods! Let's say we want an "introduction" method:
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color

   def introduction(self):
        intro = f"My name is {self.name} and my favorite color is {self.favorite_color}."
        print(intro)
```
Note that in the "introduction" method above, we aren't passing any parameters other than `self` (the class instance itself). But since `self` has attributes like `name` and `color`, we can make our intro! This would look like this:
```python
mina = Person(name = "Mina", favorite_color = "green")
mina.introduction()  # prints "My name is Mina and my favorite color is green."
```

We *can* also add parameters to specific methods! Let's say we want to *optionally* have a special way of saying "goodbye"
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color

     def introduction(self, special_goodbye=None):
          intro = f"My name is {self.name} and my favorite color is {self.favorite_color}."
          
          if special_goodbye: # add extra to outro
               intro += f" {special_goodbye}"
          
          print(intro)
```

When using the `introduction` method, we can now optionally define our special outro:
```python
mina.introduction(special_goodbye="Peace out!")  # prints "My name is Mina and my favorite color is Green. Peace out!"
```
:::


#### Solution

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------
    """
    def __init__(
        self,
        model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct",
        temperature: float = None,
    ):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self, task = "text-generation"):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        
        self.model = pipeline(
            task=task,
            model=self.model_id,
            tokenizer=self.tokenizer,
            temperature=self.temperature,
        )

### Your Turn: Lazy Loading
After having created your `load` function, consider firstly: 

:::{admonition} QUESTION
:class: red
Why do we define `task` as a parameter in `load` and not as a class attribute?

<details>
<summary>ANSWER</summary>
Firstly, you could argue that <code>task</code> is not an intrinsic property of the Chatbot class like <code>model_id</code> is. 

Secondly ...

</details>
:::

Now, let's make our `load` function a bit smarter:
:::{admonition} HANDS-ON
:class: red
Let's make our `load` function a **lazy loader**:
- Add conditions to both the loading of `tokenizer` and `pipeline`, so that the lines are *only* run if `self.tokenizer` and `self.model` is None.
:::

#### Solution

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history.

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------
    """

    def __init__(
        self,
        model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct",
        temperature: float = None,
    ):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self, task: str = "text-generation"):
        """
        Lazy loading of tokenizer and model pipeline. 
        """
        if self.tokenizer is None:
            print("[INFO:] Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        
        if self.model is None:
            print(f"[INFO:] Loading '{task}' pipeline for model {self.model_id}...")
            self.model = pipeline(
                task=task,
                model=self.model_id,
                tokenizer=self.tokenizer,
                temperature=self.temperature
            )

# trying it out
chatbot = Chatbot()
chatbot.load()

[INFO:] Loading tokenizer...
[INFO:] Loading 'text-generation' pipeline for model HuggingFaceTB/SmolLM2-360M-Instruct...


Device set to use mps:0


## 2.4 Defining a Response Function with Chat History
For the chat functionality, most LLMs run via `transformers` require a chat history formatted as a *list* of *dictionaries*:
```{figure} ../figures/class6/messages.jpg
---
name: chat_history
width: 70%
---
*Assistant* refers to the LLM’s responses, while *user* refers to our messages. The `system prompt` may be optional, as some LLMs either don't need it or have a built-in version.
``` 

Let's imagine a chat method called `generate_response` that takes a `user_input` as string. We need to format the user input, pass it to chat_history, then get a response from the LLM and format that nicely as well:

```{figure} ../figures/class6/chat_history_1_interaction_long.jpg
---
name: chat_first_interaction
width: 100%
---
The first interaction loop (without a system prompt). Start at step (1) and follow!
```


:::{admonition} LLM FRAMING: What do you mean by "most LLMs via transformers" 
:class: dropdown, fuchsia
There are many different ways to run LLM inference. Popular libraries include [vLLM](https://docs.vllm.ai/en/latest/) (faster inference on GPU) and [Ollama](https://ollama.com/) (run larger models by running [quantized](https://huggingface.co/docs/optimum/en/concept_guides/quantization) versions).

While most libraries tend to follow the same standards, sometimes there are small differences in how the LLMs are configured (e.g., `temperature` being called `temp`) or how they expect to receive input and output.

> Sidenote: I highly recommend checking both `vLLM` and `Ollama` out!
:::

### Your Turn: Create `generate_response()`
:::{admonition} HANDS-ON
:class: red
Add a `generate_response` to `Chatbot`! It should take these parameters:
1. `user_input` which is a string message written by the user
2. `max_new_tokens`, so the user can specify how long the response should be

Within the function, it should:
1. Load the chatbot 
    - Our lazy loading ensures it does not trigger if already loaded (e.g., for 2nd round of interactions)
2. Take a `user_input` and format it properly as `user_msg`
3. Generate an LLM `response` with the formatted `user_input` using the `self.model` pipeline
4. Update the Chatbots chat_history with both the user input and LLM response in chronological order
5. Return the only the content of the LLM response! 

How do you get the LLM response in the correct formats? See hints below
:::

:::{admonition} HINT 1: Get LLM response 
:class: tip, dropdown
Try to print the response to see how the model pipeline returns the chat to see if you can figure out how to:
-  Present only the generated content to the user
-  Pass the entire the formatted `assistant_msg` to the chat_history

If printing does not help you, see the next hint :)
:::

:::{admonition} HINT 2: Get LLM response
:class: tip, dropdown
When you print the response from the pipeline function, you'll get both the user message and the LLM-generated msg wrapped in a list:
```python 
print(response)
# [{'generated_text': [{'role': 'user', 'content': 'Hello. My name is Mina. My favourite color is green.'}, {'role': 'assistant', 'content': "..."}]}]
```

To get rid of the list `[]`, we can go one step in by writing `[0]`:

```python
print(response[0])
# {'generated_text': [{'role': 'user', 'content': 'Hello. My name is Mina. My favourite color is green.'}, {'role': 'assistant', 'content': '...'}]}
```

We select the key `generated_text` to get further, 
```python
print(response[0]["generated_text"])
# [{'role': 'user', 'content': 'Hello. My name is Mina. My favourite color is green.'}, {'role': 'assistant', 'content': '...'}]
```
Now, since we're interested in the LATEST message in this thread, we need to select the last entry in the list with `[-1]`:
```python
assistant_msg = response[0]["generated_text"][-1]
print(assistant_msg)
# {'role': 'assistant', 'content': 'Mina loves green, the color of the beautiful forest and the vibrant flowers that bloom during spring.'}]
```
The assistant message above is how it should be formatted for the chat_history!

To get only the message written by the assistant, we select the "content" key from the dictionary:
```python
assistant_output = response[0]["generated_text"][-1]["content"]
print(assitant_output)
```
:::

#### Solution

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to HuggingFaceTB/SmolLM2-360M-Instruct if no model_id is provided.
    -------------
    """
    def __init__(
        self,
        model_id: str = "HuggingFaceTB/SmolLM2-360M-Instruct",
        temperature: float = None,
    ):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self, task: str = "text-generation"):
        """
        Lazy loading of tokenizer and model pipeline. 
        """
        print(f"[INFO:] Loading '{task}' pipeline for model {self.model_id}...")
        if self.tokenizer is None:
            print("[INFO:] Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        
        if self.model is None:
            print("[INFO:] Loading model pipeline...")
            self.model = pipeline(
                task=task,
                model=self.model_id,
                tokenizer=self.tokenizer,
                temperature=self.temperature
            )

    def generate_response(self, user_input: str, max_new_tokens: int = 100) -> str:
        """
        Generate a response from the chatbot given user input.
        """
        # ensure the model is loaded
        self.load()

        # format user message and add to chat history
        user_msg = {"role": "user", "content": user_input}
        self.chat_history.append(user_msg)
        
        # get response
        response = self.model(self.chat_history, max_new_tokens=max_new_tokens)

        # get only the "content" to display to user 
        assistant_output = response[0]["generated_text"][-1]["content"]

        # get assitant message (we can get this from response)
        assistant_msg = response[0]["generated_text"][-1] # last message in response
        self.chat_history.append(assistant_msg)

        return assistant_output
    
# --- USAGE --- 
chatbot = Chatbot()

# generate first response
response = chatbot.generate_response("Hello. My name is Mina. My favourite color is green.")
print(response)

# generate second response
response = chatbot.generate_response("Tell me something about me.")
print(response)

[INFO:] Loading 'text-generation' pipeline for model HuggingFaceTB/SmolLM2-360M-Instruct...
[INFO:] Loading tokenizer...
[INFO:] Loading model pipeline...


Device set to use mps:0


Mina, welcome to our language learning platform. I'm happy to assist you with your color preferences. My name is SmolLM, and I'm a natural-like language model. So, what is your favorite color?
[INFO:] Loading 'text-generation' pipeline for model HuggingFaceTB/SmolLM2-360M-Instruct...
Mina, a good morning. Your name is Mina and your age is 9 years old. Your name is also a great start, but you can try to learn and learn new things to grow and develop your vocabulary.


## 2.5 Chatting with our Bot
In my solution above, I firstly wrote: 
> "Hello. My name is Mina. My favourite color is green."

In the second round of interactions, I was trying to see if it could recall something from our first interation (in the chat history):
> "Tell me something about me."

If your `generate_response` function worked as mine, your chat_history should also look like mine (with alternating user versus assistant messages!):

In [111]:
chatbot.chat_history

[{'role': 'user',
  'content': 'Hello. My name is Mina. My favourite color is green.'},
 {'role': 'assistant',
  'content': "Mina, welcome to our language learning platform. I'm happy to assist you with your color preferences. My name is SmolLM, and I'm a natural-like language model. So, what is your favorite color?"},
 {'role': 'user', 'content': 'Tell me something about me.'},
 {'role': 'assistant',
  'content': 'Mina, a good morning. Your name is Mina and your age is 9 years old. Your name is also a great start, but you can try to learn and learn new things to grow and develop your vocabulary.'}]

### Your Turn: Try chatting!
:::{admonition} HANDS-ON
:class: red
Try to chat with the bot in seperate chunks. Does it behave well? Does it work over time?
:::

## 2.6 Food for Thought and More Work You Can Do

Firstly, as a little food for thought:
:::{admonition} HANDS-ON
:class: red
Try to go to [SmolLM2's Hugging Face Model Card](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct#transformers). As a comparison, go to [Qwen3-0.6B's Model Card](https://huggingface.co/Qwen/Qwen3-0.6B#quickstart). Do they implement the chatbot in a similar way? Do they use a class?

<details>
<summary>ANSWER</summary>
SmolLM2 do not make a class, while Qwen3 do! What they both have in common is the use of the <code>AutoModelForCausalLM</code> which leads to model.generate() instead of passing text directly to a <code>pipeline</code>. 

<br><br>
While <code>pipeline</code> is a more high-level function, <code>AutoModelForCausalLM</code> gives you customizability! We didn't need it today, but it might be useful in the future!
</details>
:::

Now, there are many things you could consider to add to our Chatbot. See for example: 
:::{admonition} OPTIONAL HANDS-ON
:class: red
- Maybe we could add a `system_prompt`? Consider ways of doing this! (Or ask me!)
- Move away from the `pipeline()` and try to implement `load` and `generate_response` with `AutoModelForCausalLM` instead
- Consider how you would add more hyperparameters beyond `temperature`
    - It may make sense to group them as a dictionary that can be unpacked (e.g., **sampling_parameters), see hint!
    - If you use `AutoModelForCausalLM` instead of pipeline, you can also use `Generation_config`
:::

:::{admonition} Unpacking dictionaries into functions?
:class: tip, dropdown
Let's say we have a greet function:
```python
def greet(name, greeting):
    print(f"{greeting}, {name}!")
```

Instead of passing these directly, we can define a params dictionary and unpack:
```python
params = {"name": "Mina", "greeting": "Hello"}
greet(**params)
# Hello, Mina!
```

When you write `greet(**params)`, what python is interpreting is this:
```python
greet(name=params["name"], greeting=params["greeting"])
```

Therefore you could also write them in a different order:
```python
other_way_params = {"greeting": "Hello", "name": "Mina"} greet(**other_way_params)
```

[Source](https://docs.python.org/3/tutorial/controlflow.html#unpacking-argument-lists)
:::